# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [20]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [21]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [22]:
# TODO
vendors = pd.json_normalize(vendors_json)
merged = pd.merge(lines, vendors, on='vendor_id', how='left', indicator=True)
print(merged)
print('')
unmatched = merged[merged['_merge'] == 'left_only']
print(unmatched)
print('')
print(unmatched['vendor_id'])



           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only

      item  qty order vendor_id name zone     _merge
3  Hot Dog    3     3      V-07  NaN  NaN  left_only

3    V-07
Name: vendor_id, dtype: object


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [23]:
# TODO
merged['vendor_name'] = merged['name'].fillna('Unmatched')
byvendor = merged.groupby('vendor_name', dropna=False)['qty'].sum()
byvendor

,qty
vendor_name,
Cav Merch,1
Hoos Burgers,3
Unmatched,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [24]:
# TODO
byzone = merged.groupby('zone', dropna=False)['qty'].sum()
byzone

,qty
zone,
A,4
NaN,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [25]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

safe = [{**o, 'lines': o.get('lines') or []} for o in ragged_json]

# TODO
ragged = pd.json_normalize(safe, record_path='lines', meta=['order', 'vendor_id'])
ragged['qty'] = ragged['qty'].astype('Int64')
ragged['qty_missing'] = ragged['qty'].isna()

ragged

,item,qty,order,vendor_id,qty_missing
0,Soda,2,4,V-01,False
1,Fries,<NA>,6,V-01,True


### Q6 — Validate

In [26]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) Flattening put qty into a column so you can groupby sum

b) Zero means none were sold and missing means you have no idea what the value is.